# FSboard P3 benchmark launcher

Run the cells from top to bottom. When asked, upload `colab/p3_source_bundle.zip` from this repository. The bundle contains benchmark code, the 126D checkpoint, and saved reference predictions; it contains **no FSboard NPZ data**.

The benchmark authenticates to Drive, verifies actual media access with an authenticated 4-byte range read, then downloads the ZIP to `/content/fsboard_cache/`, checks its Drive MD5 and SHA-256, and extracts only the test NPZ. The 24-sample smoke gate is compared directly with the reference artifact. Full evaluation starts only if the smoke labels and predictions both match exactly.

Drive file ID used by cells 17 and 19 of the original notebook: `1dhcrbZeb3-5IshIkzatVmNO3MOTVFjJW`.

In [ ]:
from google.colab import files
from pathlib import Path
import tempfile
import zipfile

uploaded = files.upload()  # Select colab/p3_source_bundle.zip from this repository.
if len(uploaded) != 1:
    raise RuntimeError(f"Upload exactly one source bundle ZIP; received {len(uploaded)} files.")

bundle_name, bundle_bytes = next(iter(uploaded.items()))
if not bundle_name.lower().endswith(".zip"):
    raise RuntimeError(f"Expected a .zip source bundle, got {bundle_name!r}.")
bundle_path = Path("/content") / Path(bundle_name).name
bundle_path.write_bytes(bundle_bytes)
extract_root = Path(tempfile.mkdtemp(prefix="fsboard_p3_source_"))
with zipfile.ZipFile(bundle_path) as bundle:
    bad_member = bundle.testzip()
    if bad_member:
        raise IOError(f"Corrupt source bundle member: {bad_member}")
    members = bundle.infolist()
    if any(Path(item.filename).is_absolute() or ".." in Path(item.filename).parts for item in members):
        raise ValueError("Source bundle contains an unsafe member path.")
    bundle.extractall(extract_root)

REPO_ROOT = extract_root / "fingerspelling-to-text"
if not (REPO_ROOT / "scripts" / "colab_benchmark.py").is_file():
    raise FileNotFoundError("The uploaded ZIP is not the FSboard P3 source bundle.")
print(f"Source bundle ready at {REPO_ROOT}; uploaded {len(bundle_bytes):,} bytes.")

In [ ]:
import os
os.chdir(REPO_ROOT)
%pip install -e ".[colab]"

## Run the gated benchmark

Authorize the Drive mount/API prompt when it appears. The script first prints the 24-sample metrics and direct reference comparison. It proceeds to all 12,515 samples only when the smoke gate passes; otherwise it saves the mismatch report to Drive and stops.

In [ ]:
!python scripts/colab_benchmark.py --source-archive-file-id 1dhcrbZeb3-5IshIkzatVmNO3MOTVFjJW --smoke-count 24 --batch-size 16 --device auto